In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow as tf

In [3]:
dataset = load_dataset("dair-ai/emotion")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [4]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [5]:
def tokenize_data(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_data)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
train_dataset = tokenized_dataset["train"].shuffle(1000).batch(32)
val_dataset = tokenized_dataset["validation"].batch(32)
test_dataset = tokenized_dataset["test"].batch(32)

Batching examples:   0%|          | 0/16000 [00:00<?, ? examples/s]

Batching examples:   0%|          | 0/2000 [00:00<?, ? examples/s]

Batching examples:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
model = TFDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=6,from_pt = True
)

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSeq

In [9]:
import tf_keras
model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=2e-5),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [10]:
model.summary()

Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  4614      
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
Total params: 66958086 (255.42 MB)
Trainable params: 66958086 (255.42 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
from transformers import DefaultDataCollator

# Data collator handles padding dynamically
data_collator = DefaultDataCollator(return_tensors="tf")

# Convert Hugging Face dataset → tf.data.Dataset
tf_train_dataset = model.prepare_tf_dataset(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator,
)

tf_val_dataset = model.prepare_tf_dataset(
    tokenized_dataset["validation"],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)
tf_test_dataset = model.prepare_tf_dataset(
    tokenized_dataset["test"],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)

In [12]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=10
)

Epoch 1/10
500/500 [==============================] - 245s 438ms/step - loss: 0.5519 - accuracy: 0.8108 - val_loss: 0.1831 - val_accuracy: 0.9265
Epoch 2/10
500/500 [==============================] - 207s 415ms/step - loss: 0.1539 - accuracy: 0.9374 - val_loss: 0.1352 - val_accuracy: 0.9420
Epoch 3/10
500/500 [==============================] - 208s 415ms/step - loss: 0.1085 - accuracy: 0.9508 - val_loss: 0.1268 - val_accuracy: 0.9415
Epoch 4/10
500/500 [==============================] - 206s 412ms/step - loss: 0.0883 - accuracy: 0.9597 - val_loss: 0.1391 - val_accuracy: 0.9360
Epoch 5/10
500/500 [==============================] - 206s 413ms/step - loss: 0.0738 - accuracy: 0.9655 - val_loss: 0.1762 - val_accuracy: 0.9340
Epoch 6/10
500/500 [==============================] - 206s 411ms/step - loss: 0.0638 - accuracy: 0.9718 - val_loss: 0.1830 - val_accuracy: 0.9365
Epoch 7/10
500/500 [==============================] - 206s 413ms/step - loss: 0.0544 - accuracy: 0.9759 - val_loss: 0.1638 -

In [18]:
import tf_keras
model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=2e-8),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [19]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=20,
    initial_epoch=10,
)

Epoch 11/20
500/500 [==============================] - 246s 448ms/step - loss: 0.0295 - accuracy: 0.9898 - val_loss: 0.2163 - val_accuracy: 0.9360
Epoch 12/20
500/500 [==============================] - 208s 416ms/step - loss: 0.0297 - accuracy: 0.9904 - val_loss: 0.2148 - val_accuracy: 0.9355
Epoch 13/20
500/500 [==============================] - 206s 412ms/step - loss: 0.0272 - accuracy: 0.9906 - val_loss: 0.2136 - val_accuracy: 0.9355
Epoch 14/20
500/500 [==============================] - 206s 413ms/step - loss: 0.0267 - accuracy: 0.9909 - val_loss: 0.2123 - val_accuracy: 0.9360
Epoch 15/20
500/500 [==============================] - 207s 413ms/step - loss: 0.0266 - accuracy: 0.9908 - val_loss: 0.2111 - val_accuracy: 0.9365
Epoch 16/20
500/500 [==============================] - 206s 413ms/step - loss: 0.0246 - accuracy: 0.9912 - val_loss: 0.2101 - val_accuracy: 0.9360
Epoch 17/20
500/500 [==============================] - 206s 412ms/step - loss: 0.0259 - accuracy: 0.9919 - val_loss: 0

In [25]:
model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=3e-5),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [26]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=25,
    initial_epoch=20,
)

Epoch 21/25
500/500 [==============================] - 244s 445ms/step - loss: 0.0423 - accuracy: 0.9854 - val_loss: 0.2209 - val_accuracy: 0.9370
Epoch 22/25
500/500 [==============================] - 208s 416ms/step - loss: 0.0276 - accuracy: 0.9910 - val_loss: 0.2516 - val_accuracy: 0.9375
Epoch 23/25
500/500 [==============================] - 205s 411ms/step - loss: 0.0311 - accuracy: 0.9893 - val_loss: 0.2563 - val_accuracy: 0.9375
Epoch 24/25
500/500 [==============================] - 206s 412ms/step - loss: 0.0197 - accuracy: 0.9936 - val_loss: 0.2865 - val_accuracy: 0.9325
Epoch 25/25
500/500 [==============================] - 205s 410ms/step - loss: 0.0226 - accuracy: 0.9937 - val_loss: 0.2429 - val_accuracy: 0.9360


In [27]:
model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=3e-8),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [28]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=30,
    initial_epoch=25,
)

Epoch 26/30
500/500 [==============================] - 247s 451ms/step - loss: 0.0138 - accuracy: 0.9961 - val_loss: 0.2429 - val_accuracy: 0.9360
Epoch 27/30
500/500 [==============================] - 208s 416ms/step - loss: 0.0131 - accuracy: 0.9961 - val_loss: 0.2428 - val_accuracy: 0.9360
Epoch 28/30
500/500 [==============================] - 206s 412ms/step - loss: 0.0122 - accuracy: 0.9965 - val_loss: 0.2428 - val_accuracy: 0.9365
Epoch 29/30
500/500 [==============================] - 206s 412ms/step - loss: 0.0133 - accuracy: 0.9957 - val_loss: 0.2429 - val_accuracy: 0.9360
Epoch 30/30
500/500 [==============================] - 206s 411ms/step - loss: 0.0122 - accuracy: 0.9964 - val_loss: 0.2429 - val_accuracy: 0.9360


In [29]:
loss, accuracy = model.evaluate(tf_train_dataset)
print(f"Train Accuracy: {accuracy*100:.2f}%")

500/500 [==============================] - 71s 142ms/step - loss: 0.0083 - accuracy: 0.9973
Train Accuracy: 99.73%


In [30]:
loss, accuracy = model.evaluate(tf_val_dataset)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

63/63 [==============================] - 9s 142ms/step - loss: 0.2429 - accuracy: 0.9360
Validation Accuracy: 93.60%


In [31]:
loss, accuracy = model.evaluate(tf_test_dataset)
print(f"Test Accuracy: {accuracy*100:.2f}%")

63/63 [==============================] - 9s 141ms/step - loss: 0.2956 - accuracy: 0.9270
Test Accuracy: 92.70%


In [32]:
model.save_pretrained("/content/drive/MyDrive/emotion_model")
tokenizer.save_pretrained("/content/drive/MyDrive/emotion_model")

('/content/drive/MyDrive/emotion_model/tokenizer_config.json',
 '/content/drive/MyDrive/emotion_model/special_tokens_map.json',
 '/content/drive/MyDrive/emotion_model/vocab.txt',
 '/content/drive/MyDrive/emotion_model/added_tokens.json',
 '/content/drive/MyDrive/emotion_model/tokenizer.json')

In [33]:
labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="tf", truncation=True, padding=True, max_length=128)
    outputs = model(inputs)
    logits = outputs.logits
    predicted_class = int(tf.argmax(logits, axis=1))
    emotion = labels[predicted_class]
    return emotion

# Test examples
print("Text: I am so happy today!")
print("Predicted Emotion:", predict_emotion("I am so happy today!"))

print("Text: I am scared of failing my exam.")
print("Predicted Emotion:", predict_emotion("I am scared of failing my exam."))

Text: I am so happy today!
Predicted Emotion: joy
Text: I am scared of failing my exam.
Predicted Emotion: fear
